# Digital Wallet — load test

Kaggle settings: **Accelerator = None (CPU)**, **Internet = On**.
For the full set (about 1¼ hours) use **Save Version → Save & Run All**.

The app + MySQL run on CPUs 0,2 (physical core 0), k6 on CPUs 1,3 (physical core 1),
so k6 never takes CPU from the app.

In [ ]:
REPO_URL = 'https://github.com/LeonaRaging/SAUETDigitalWallet.git'
BRANCH   = 'main'    # app version to test (branch or tag)
LABEL    = BRANCH    # name of the results folder

## Setup

In [ ]:
import os, subprocess

REPO = '/tmp/repo'
S    = f'{REPO}/digitalwallet-stress-test/scripts'   # install, MySQL, build, start app
T    = f'{REPO}/digitalwallet-stress-test/v2'        # the k6 tests
OUT  = f'/kaggle/working/results/{LABEL}'
os.makedirs(OUT, exist_ok=True)
os.environ['SUT_DIR'] = REPO
os.environ['JAVA_TOOL_OPTIONS'] = '-XX:ActiveProcessorCount=2'   # the app gets 2 CPUs

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

def fresh_app():
    """Empty database + restarted app, both pinned to CPUs 0,2."""
    sh(f'bash {S}/90_reset_db.sh > /dev/null && bash {S}/30_start_app.sh > /dev/null')
    sh('taskset -a -cp 0,2 $(pgrep -x mysqld) > /dev/null && taskset -a -cp 0,2 $(cat /tmp/app.pid)')

def k6(script, name, **settings):
    """Run a k6 test on CPUs 1,3; keeps its summary as <name>.json and <name>.txt."""
    e = ' '.join(f'-e {k}={v}' for k, v in settings.items())
    subprocess.run(f'taskset -c 1,3 k6 run --quiet {e} --summary-export {OUT}/{name}.json '
                   f'{T}/{script} 2>&1 | tee {OUT}/{name}.txt', shell=True)

def money_check():
    sh(f'mysql -uroot -D digitalwallet_db -t < {T}/money_check.sql')

sh(f'rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO}')
sh(f"find {REPO} -name '*.sh' -exec sed -i 's/\\r$//' {{}} +")          # Windows line endings
sh(f'bash {S}/00_install.sh && bash {S}/10_mysql.sh && bash {S}/20_build_sut.sh')

## Smoke test (1 min) — check everything works before the long runs

In [ ]:
fresh_app()
k6('load.js', 'smoke', RATE=5, DURATION='1m', USERS=20)

## Load test — 100 requests/s for 10 min, then check no money was created or lost

In [ ]:
fresh_app()
k6('load.js', 'load_100', RATE=100, DURATION='10m')
money_check()

## Hot wallet — everyone pays the same wallet (worst case for locking; shows the I2 fix)

In [ ]:
fresh_app()
k6('load.js', 'hot_100', RATE=100, DURATION='5m', HOT=1)
money_check()

## Long history — every user starts with 200 past transactions (shows pagination / caching, I3)

In [ ]:
fresh_app()
k6('load.js', 'history_100', RATE=100, DURATION='5m', HISTORY=200)
money_check()

## Stress test — the same test at higher and higher rates, 2 min each

In [ ]:
RATES = [100, 200, 300, 400, 500, 600, 800]
for rate in RATES:
    fresh_app()
    k6('load.js', f'stress_{rate}', RATE=rate, DURATION='2m')

## UC1 (same request 20× → one debit) and UC2 (double spend → at most one succeeds)

In [ ]:
fresh_app()
k6('uc1.js', 'uc1')
fresh_app()
k6('uc2.js', 'uc2')

## Results table

In [ ]:
import json

def m(name, metric, field, default=0):
    try:
        return json.load(open(f'{OUT}/{name}.json'))['metrics'].get(metric, {}).get(field, default)
    except FileNotFoundError:
        return None

print(f"{'test':<12}{'target/s':>9}{'achieved/s':>11}{'p95 ms':>9}{'failed':>9}{'409s':>7}{'400s':>7}")
runs = [('load_100', 100, 600), ('hot_100', 100, 300), ('history_100', 100, 300)]
for name, rate, secs in runs + [(f'stress_{r}', r, 120) for r in RATES]:
    it = m(name, 'iterations', 'count')
    if it is None:
        continue
    print(f"{name:<12}{rate:>9}{it / secs:>11.1f}{m(name, 'http_req_duration{scenario:load}', 'p(95)'):>9.0f}"
          f"{m(name, 'http_req_failed{scenario:load}', 'value'):>9.2%}{m(name, 'conflicts_409', 'count'):>7}"
          f"{m(name, 'rejected_400', 'count'):>7}")

print('\nUC1 rounds debited once:', m('uc1', 'rounds_debited_once', 'count'), '/ 50,',
      'extra debits:', m('uc1', 'extra_debits', 'count'))
print('UC2 both succeeded:', m('uc2', 'both_succeeded', 'count'), '| wrong balance:', m('uc2', 'wrong_balance', 'count'),
      '| loser got 400:', m('uc2', 'loser_got_400', 'count'), '| loser got 409:', m('uc2', 'loser_got_409', 'count'))

Download the zip from the Output tab and put it in `digitalwallet-docs/stress-test/results/<LABEL>/`.

In [ ]:
import shutil
shutil.make_archive(OUT, 'zip', OUT)